# Carga de arquivos CSV para banco SQLite

Este notebook é autocontido. Configure os caminhos na célula abaixo e execute as células em ordem.

**Etapas:**
1. Escanear a pasta de entrada por arquivos CSV.
2. Derivar o nome da tabela SQLite a partir do nome de cada arquivo.
3. Inferir e otimizar os tipos de dados de cada coluna (downcast automático).
4. Gravar cada arquivo como uma tabela no banco de saída.
5. Exibir sumário e validar contagem de linhas.

In [1]:
from __future__ import annotations

import re
import sqlite3
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)

In [2]:
INPUT_DIR     = "/home/carlos_rocha/Documentos/IBGE/ibge-censo-agro-research/data_example/samples"   # Pasta com os arquivos CSV de entrada
OUTPUT_DB     = "/home/carlos_rocha/Documentos/IBGE/ibge-censo-agro-research/data_example/amstr_geral.db"   # Caminho de saída do banco SQLite
CSV_ENCODING  = "utf-8"
CSV_SEPARATOR = ","
OVERWRITE_DB  = True   # Se True, recria o banco do zero a cada execução

# --- não edite abaixo desta linha ---
assert INPUT_DIR is not None, "Configure INPUT_DIR antes de executar."
assert OUTPUT_DB is not None, "Configure OUTPUT_DB antes de executar."

_INPUT_DIR = Path(INPUT_DIR)
_OUTPUT_DB = Path(OUTPUT_DB)

assert _INPUT_DIR.exists(), f"Diretório de entrada não encontrado: {_INPUT_DIR}"

print(f"Entrada : {_INPUT_DIR.resolve()}")
print(f"Saída   : {_OUTPUT_DB.resolve()}")
print(f"Sobrescrever banco existente: {OVERWRITE_DB}")

Entrada : /home/carlos_rocha/Documentos/IBGE/ibge-censo-agro-research/data_example/samples
Saída   : /home/carlos_rocha/Documentos/IBGE/ibge-censo-agro-research/data_example/amstr_geral.db
Sobrescrever banco existente: True


In [3]:
def derive_table_name(path: Path) -> str:
    """Sanitizes a CSV filename into a valid SQLite table name."""
    name = re.sub(r"[^a-z0-9]", "_", path.stem.lower()).strip("_")
    if name and name[0].isdigit():
        name = "t_" + name
    return name or "table"


def quote_ident(name: str) -> str:
    return '"' + str(name).replace('"', '""') + '"'


def memory_mb(df: pd.DataFrame) -> float:
    return float(df.memory_usage(deep=True).sum() / 1024 ** 2)


def table_row_count(conn: sqlite3.Connection, table_name: str) -> int:
    return int(conn.execute(f"SELECT COUNT(*) FROM {quote_ident(table_name)}").fetchone()[0])


def _infer_smallest_nullable_int(series: pd.Series) -> str:
    non_null = series.dropna()
    if non_null.empty:
        return "Int8"
    lo, hi = int(non_null.min()), int(non_null.max())
    for name, info in [("Int8", np.int8), ("Int16", np.int16), ("Int32", np.int32), ("Int64", np.int64)]:
        if lo >= np.iinfo(info).min and hi <= np.iinfo(info).max:
            return name
    return "Int64"


def _can_cast_float32_losslessly(series: pd.Series) -> bool:
    arr = series.dropna().astype(np.float64).to_numpy()
    return arr.size == 0 or np.array_equal(arr, arr.astype(np.float32).astype(np.float64))


def _infer_text_dtype(series: pd.Series) -> pd.Series:
    as_str = series.astype("string")
    non_null = as_str.dropna()
    if non_null.empty:
        return as_str
    n_unique = int(non_null.nunique())
    if n_unique <= 256 or n_unique / len(non_null) <= 0.20:
        return as_str.astype("category")
    return as_str


def auto_downcast(series: pd.Series) -> tuple[pd.Series, str]:
    """Tries numeric downcast first; falls back to category/string.
    Returns (optimized_series, final_dtype_name)."""
    converted = pd.to_numeric(series, errors="coerce")
    if (series.notna() & converted.isna()).any():
        result = _infer_text_dtype(series)
        return result, str(result.dtype)
    non_null = converted.dropna().astype(np.float64)
    if non_null.empty:
        return converted.astype("Float32"), "Float32"
    if np.isclose(np.mod(non_null.to_numpy(), 1), 0).all():
        dtype = _infer_smallest_nullable_int(non_null)
        return converted.round().astype(dtype), dtype
    if _can_cast_float32_losslessly(non_null):
        return converted.astype("Float32"), "Float32"
    return converted.astype("Float64"), "Float64"


def prepare_for_sqlite(df: pd.DataFrame) -> pd.DataFrame:
    """Converts nullable types to SQLite-compatible types."""
    out = df.copy()
    for col in out.columns:
        if str(out[col].dtype) == "category":
            out[col] = out[col].astype("string")
        if pd.api.types.is_string_dtype(out[col]):
            out[col] = out[col].astype(object)
    return out


print("Funções carregadas.")

Funções carregadas.


In [4]:
csv_files = sorted(_INPUT_DIR.glob("*.csv"))
assert csv_files, f"Nenhum arquivo CSV encontrado em: {_INPUT_DIR}"

inventory = pd.DataFrame([
    {
        "arquivo": p.name,
        "tabela_destino": derive_table_name(p),
        "tamanho_kb": round(p.stat().st_size / 1024, 1),
    }
    for p in csv_files
])
print(f"{len(csv_files)} arquivo(s) encontrado(s):")
display(inventory)

3 arquivo(s) encontrado(s):


,arquivo,tabela_destino,tamanho_kb
0,amstr_estabel.csv,amstr_estabel,1.0
1,amstr_lav_temp.csv,amstr_lav_temp,0.3
2,amstr_silv.csv,amstr_silv,0.3


In [5]:
if OVERWRITE_DB and _OUTPUT_DB.exists():
    _OUTPUT_DB.unlink()

_OUTPUT_DB.parent.mkdir(parents=True, exist_ok=True)

summary_rows: list[dict[str, Any]] = []
dtype_log_rows: list[dict[str, Any]] = []

with sqlite3.connect(_OUTPUT_DB) as conn:
    for csv_path in csv_files:
        table_name = derive_table_name(csv_path)
        df = pd.read_csv(
            csv_path, encoding=CSV_ENCODING, sep=CSV_SEPARATOR, low_memory=False
        )

        # Removendo todas as colunas sem nome (ex: "Unnamed: 0") para evitar problemas de memória e SQLite
        unnamed_cols = [col for col in df.columns if col.startswith("Unnamed:")]
        if unnamed_cols:
            print(f"Removendo colunas sem nome de {csv_path.name}: {unnamed_cols}")
            df.drop(columns=unnamed_cols, inplace=True)

        mem_before = memory_mb(df)
        rows_in = len(df)

        for col in df.columns:
            orig_dtype = str(df[col].dtype)
            df[col], final_dtype = auto_downcast(df[col])
            dtype_log_rows.append({
                "tabela": table_name,
                "coluna": col,
                "dtype_original": orig_dtype,
                "dtype_final": final_dtype,
            })

        mem_after = memory_mb(df)
        prepare_for_sqlite(df).to_sql(table_name, conn, if_exists="replace", index=False)
        rows_out = table_row_count(conn, table_name)

        reducao = (mem_before - mem_after) / mem_before * 100 if mem_before > 0 else 0.0
        summary_rows.append({
            "arquivo": csv_path.name,
            "tabela": table_name,
            "linhas_csv": rows_in,
            "linhas_sqlite": rows_out,
            "colunas": len(df.columns),
            "mem_antes_mb": round(mem_before, 3),
            "mem_depois_mb": round(mem_after, 3),
            "reducao_pct": round(reducao, 1),
        })

    conn.commit()

print(f"Banco gerado: {_OUTPUT_DB.resolve()}")

Removendo colunas sem nome de amstr_estabel.csv: ['Unnamed: 0']
Removendo colunas sem nome de amstr_lav_temp.csv: ['Unnamed: 0']
Removendo colunas sem nome de amstr_silv.csv: ['Unnamed: 0']
Banco gerado: /home/carlos_rocha/Documentos/IBGE/ibge-censo-agro-research/data_example/amstr_geral.db


In [6]:
summary_df = pd.DataFrame(summary_rows)
dtype_log_df = pd.DataFrame(dtype_log_rows)

print("Sumário por tabela:")
display(summary_df)

linhas_ok = (summary_df["linhas_csv"] == summary_df["linhas_sqlite"]).all()
if linhas_ok:
    print("OK — todas as tabelas mantiveram a contagem de linhas.")
else:
    print("ATENÇÃO — diferenças de contagem detectadas:")
    display(summary_df[summary_df["linhas_csv"] != summary_df["linhas_sqlite"]])

print("\nLog de tipagem por coluna:")
display(dtype_log_df.sort_values(["tabela", "coluna"]))

Sumário por tabela:


,arquivo,tabela,linhas_csv,linhas_sqlite,colunas,mem_antes_mb,mem_depois_mb,reducao_pct
0,amstr_estabel.csv,amstr_estabel,10,10,16,0.002,0.002,17.9
1,amstr_lav_temp.csv,amstr_lav_temp,3,3,11,0.000,0.000,42.1
2,amstr_silv.csv,amstr_silv,3,3,11,0.000,0.000,43.6


OK — todas as tabelas mantiveram a contagem de linhas.

Log de tipagem por coluna:


,tabela,coluna,dtype_original,dtype_final
2,amstr_estabel,NUM_FACE,int64,Int8
1,amstr_estabel,NUM_QUADRA,int64,Int8
0,amstr_estabel,V010100,int64,Int64
3,amstr_estabel,V010800,int64,Int8
4,amstr_estabel,V011300,object,category
5,amstr_estabel,V01150000,int64,Int8
6,amstr_estabel,V01150101,float64,Int64
7,amstr_estabel,V01150102,float64,Int64
8,amstr_estabel,V01170100,int64,Int16
9,amstr_estabel,V01170500,int64,Int8
